<a href="https://colab.research.google.com/github/Su-creator-spec/ADRPredict-AI-Powered-Adverse-Drug-Reaction-Prediction-Tool/blob/main/AIR_QUALITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
kunshbhatia_delhi_air_quality_dataset_path = kagglehub.dataset_download('kunshbhatia/delhi-air-quality-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
path='/kaggle/input/delhi-air-quality-dataset/final_dataset.csv'
df=pd.read_csv(path)
df.head()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#PM2.5 sns line plot
plt.figure(figsize=(12, 6))
sns.lineplot(data=df,x="Year",y="PM2.5")
plt.title("PM2.5 Over time")
plt.show()


In [ ]:
# Boxplot by working day
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Holidays_Count', y='AQI')
plt.title('AQI Distribution: Working Day vs Non-Working Day')
plt.show()

In [ ]:
#PM2.5 monthly avergaes over time
monthly_avg=df.groupby(['Month', 'Year'])[['PM2.5', 'PM10','NO2', 'SO2']].mean().unstack()
sns.heatmap(data=monthly_avg['PM2.5'],cmap='YlGnBu',annot=True,fmt='.1f')
plt.title('Monthly PM2.5 Levels')
plt.show()

In [ ]:
sns.heatmap(data=monthly_avg['NO2'],cmap='YlGnBu',annot=True,fmt='.1f')
plt.title('Monthly NO2 Levels')
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
df.set_index('Date',inplace=True)

result=seasonal_decompose(df["AQI"],model='multiplicative',period=365)
result.plot()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# selcting the features and the values to predict
features=['Holidays_Count', 'PM2.5', 'PM10','NO2', 'SO2', 'CO', 'Ozone']
X=df[features]
y=df['AQI']

# spliting the dataset
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#training the model
model=RandomForestRegressor()
model.fit(X_train,y_train)

# predicting the model
y_pred=model.predict(X_test)
print("RSME :",mean_squared_error(y_test,y_pred,squared=False))
print(model.score(X_test,y_test))